# 🔧 Data Preprocessing and Augmentation

## Document Type Classification - Part 2

**Goal**: Prepare the dataset for model training by:
- Loading and preprocessing images
- Implementing data augmentation strategies
- Creating data generators for efficient training
- Optimizing data pipeline for deep learning

**Dataset**: 2,100 real identity documents (700 per type)
- 🛂 Passports (Greece) - 700 images
- 🪪 ID Cards (Russia) - 700 images  
- 🚗 Driver's Licenses (USA - West Virginia) - 700 images

**Next Steps**: After preprocessing, we'll build and train classification models.


## 1. Setup and Imports


In [16]:
# ===========================
# 📦 Data handling
# ===========================
import pandas as pd
import numpy as np
from pathlib import Path
import os
import sklearn as sk

# ===========================
# 🖼️ Image processing
# ===========================
from PIL import Image
import cv2


# ===========================
# ⚙️ Display & Reproducibility
# ===========================
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


ImportError: cannot import name '_slice' from 'numpy._core.umath' (/Users/roy-siftt/final-project/venv/lib/python3.13/site-packages/numpy/_core/umath.py)

## 2. Load Dataset


In [ ]:
# Load the dataset
DATA_DIR = '../../datasets/idnet/document_type_classification/data/'
TRAIN_CSV = DATA_DIR + 'train_dataset.csv'
VAL_CSV = DATA_DIR + 'val_dataset.csv'
TEST_CSV = DATA_DIR + 'test_dataset.csv'

# Load CSV files
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("📊 Dataset loaded successfully!")
print(f"   Training set: {len(train_df):,} images")
print(f"   Validation set: {len(val_df):,} images")
print(f"   Test set: {len(test_df):,} images")
print(f"   Total: {len(train_df) + len(val_df) + len(test_df):,} images")

# Check class distribution
print("\n📈 Class distribution:")
print("Training set:")
print(train_df['document_type'].value_counts())
print("\nValidation set:")
print(val_df['document_type'].value_counts())
print("\nTest set:")
print(test_df['document_type'].value_counts())


## 3. Image Preprocessing Functions


In [ ]:
def load_and_preprocess_image(image_path, target_size=(224, 224)):
    """
    Load and preprocess a single image.
    
    Args:
        image_path: Path to the image file
        target_size: Target size for resizing (width, height)
    
    Returns:
        Preprocessed image as numpy array
    """
    # Load image
    img = Image.open(image_path)
    
    # Convert to RGB if needed
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    # Resize to target size
    img = img.resize(target_size)
    
    # Convert to numpy array and normalize to [0, 1]
    img_array = np.array(img) / 255.0
    
    return img_array

def create_label_mapping(df):
    """
    Create label mapping for document types.
    
    Args:
        df: DataFrame with 'document_type' column
    
    Returns:
        label_to_idx: Dictionary mapping document types to indices
        idx_to_label: Dictionary mapping indices to document types
    """
    unique_types = sorted(df['document_type'].unique())
    label_to_idx = {doc_type: idx for idx, doc_type in enumerate(unique_types)}
    idx_to_label = {idx: doc_type for doc_type, idx in label_to_idx.items()}
    
    return label_to_idx, idx_to_label

# Create label mappings
label_to_idx, idx_to_label = create_label_mapping(train_df)

print("🏷️ Label mappings created:")
print("Document type → Index:")
for doc_type, idx in label_to_idx.items():
    print(f"   {doc_type}: {idx}")

print(f"\nTotal classes: {len(label_to_idx)}")


## 4. Test Image Loading


In [ ]:
# Test loading one image
sample_path = train_df.iloc[0]['image_path']
sample_img = Image.open(sample_path)

print(f"🖼️ Sample image (before preprocessing):")
print(f"   Path: {sample_path}")
print(f"   Size: {sample_img.size}")
print(f"   Mode: {sample_img.mode}")
print(f"   Document type: {train_df.iloc[0]['document_type']}")

# Test preprocessing function
preprocessed_img = load_and_preprocess_image(sample_path)

print(f"\n✅ Preprocessing function created!")
print(f"   Preprocessed shape: {preprocessed_img.shape}")
print(f"   Data type: {preprocessed_img.dtype}")
print(f"   Value range: [{preprocessed_img.min():.3f}, {preprocessed_img.max():.3f}]")

# Display original and preprocessed images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original image
axes[0].imshow(sample_img)
axes[0].set_title(f"Original Image\n{train_df.iloc[0]['document_type']}")
axes[0].axis('off')

# Preprocessed image
axes[1].imshow(preprocessed_img)
axes[1].set_title(f"Preprocessed Image\n{preprocessed_img.shape}")
axes[1].axis('off')

plt.tight_layout()
plt.show()


## 5. Data Augmentation Strategy


In [ ]:
# Define augmentation parameters
TARGET_SIZE = (224, 224)
BATCH_SIZE = 32

# Training data augmentation (more aggressive)
train_transforms = transforms.Compose([
    transforms.Resize(TARGET_SIZE),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomAffine(degrees=0, shear=10),
    transforms.RandomAffine(degrees=0, scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test data augmentation (minimal)
val_test_transforms = transforms.Compose([
    transforms.Resize(TARGET_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("🔄 Data augmentation configured!")
print(f"   Target size: {TARGET_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Training augmentation: Enabled")
print(f"   Validation augmentation: Disabled")


## 6. Create Data Generators


In [ ]:
# Create custom dataset class
class DocumentDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
        
        # Create label mapping
        self.classes = sorted(dataframe['document_type'].unique())
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = row['image_path']
        label = self.class_to_idx[row['document_type']]
        
        # Load image
        image = Image.open(image_path).convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Create datasets
train_dataset = DocumentDataset(train_df, transform=train_transforms)
val_dataset = DocumentDataset(val_df, transform=val_test_transforms)
test_dataset = DocumentDataset(test_df, transform=val_test_transforms)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("📦 Data loaders created successfully!")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")
print(f"   Classes found: {train_dataset.classes}")
print(f"   Number of classes: {len(train_dataset.classes)}")


## 7. Visualize Augmented Data


In [ ]:
# Get a batch of augmented training data
dataiter = iter(train_loader)
batch_x, batch_y = next(dataiter)

print(f"🖼️ Augmented batch shape: {batch_x.shape}")
print(f"   Images: {batch_x.shape[0]}")
print(f"   Channels: {batch_x.shape[1]}")
print(f"   Height: {batch_x.shape[2]}")
print(f"   Width: {batch_x.shape[3]}")
print(f"   Labels shape: {batch_y.shape}")

# Display augmented images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i in range(8):
    # Get image and label
    img = batch_x[i]
    label_idx = batch_y[i].item()
    label_name = train_dataset.classes[label_idx]
    
    # Convert tensor to numpy for display
    img_np = img.permute(1, 2, 0).numpy()
    # Denormalize for display
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    
    # Display image
    axes[i].imshow(img_np)
    axes[i].set_title(f"{label_name}")
    axes[i].axis('off')

plt.suptitle("Augmented Training Images", fontsize=16)
plt.tight_layout()
plt.show()

# Show class distribution in this batch
print(f"\n📊 Class distribution in this batch:")
for i, class_name in enumerate(train_dataset.classes):
    count = (batch_y == i).sum().item()
    print(f"   {class_name}: {count} images")


## 8. Data Pipeline Summary


In [ ]:
# Summary of our data pipeline
print("📋 Data Pipeline Summary")
print("=" * 50)

print(f"\n📊 Dataset Statistics:")
print(f"   Total images: {len(train_df) + len(val_df) + len(test_df):,}")
print(f"   Training: {len(train_df):,} images")
print(f"   Validation: {len(val_df):,} images")
print(f"   Test: {len(test_df):,} images")

print(f"\n🏷️ Classes:")
for i, class_name in enumerate(train_generator.class_indices.keys()):
    print(f"   {i}: {class_name}")

print(f"\n🖼️ Image Processing:")
print(f"   Target size: {TARGET_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Normalization: [0, 1] range")
print(f"   Data type: float32")

print(f"\n🔄 Augmentation (Training only):")
print(f"   Rotation: ±15°")
print(f"   Shift: ±10% (width/height)")
print(f"   Shear: ±10%")
print(f"   Zoom: ±10%")
print(f"   Brightness: ±20%")
print(f"   Horizontal flip: No")
print(f"   Vertical flip: No")

print(f"\n📦 Data Loaders:")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

print(f"\n✅ Ready for model training!")
print(f"   Next step: Build and train classification models with PyTorch")


## 9. Save Preprocessing Configuration


In [ ]:
# Save configuration for future use
config = {
    'target_size': TARGET_SIZE,
    'batch_size': BATCH_SIZE,
    'random_state': RANDOM_STATE,
    'class_indices': {cls: idx for idx, cls in enumerate(train_dataset.classes)},
    'num_classes': len(train_dataset.classes),
    'augmentation_params': {
        'rotation_range': 15,
        'translation_range': 0.1,
        'shear_range': 10,
        'scale_range': 0.1,
        'brightness_range': 0.2,
        'horizontal_flip': False,
        'vertical_flip': False
    }
}

# Save to file
import json
config_path = '../../models/preprocessing_config.json'
os.makedirs(os.path.dirname(config_path), exist_ok=True)

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"💾 Configuration saved to: {config_path}")
print(f"   Target size: {config['target_size']}")
print(f"   Batch size: {config['batch_size']}")
print(f"   Number of classes: {config['num_classes']}")
print(f"   Classes: {list(config['class_indices'].keys())}")

print(f"\n🎯 Next Steps:")
print(f"   1. Run this notebook to test the data pipeline")
print(f"   2. Create model training notebook (03_baseline_model.ipynb)")
print(f"   3. Use the saved configuration for consistent preprocessing")
